# Cora centroid hypergraph — HMRW sweep over `q`

Loads **`data/cora_centroid/cora_centroid.pickle`** (hypergraph + labels; produced by `cora_centroid_hypergraph.ipynb`) and runs **HMRW** (`src.utils.hmrw_embeddings`) with most parameters fixed, varying only **`q`**. No Cora source folder is needed on read.

Evaluation: **stratified 5-fold CV** with logistic regression on the learned embeddings.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hypergraphx import Hypergraph
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.utils import hmrw_embeddings

In [2]:
def evaluate_embeddings(X, y, seed=42, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    clf = LogisticRegression(max_iter=3000)
    scores = cross_validate(
        clf,
        X,
        y,
        cv=cv,
        scoring={"micro_f1": "f1_micro", "macro_f1": "f1_macro"},
        n_jobs=-1,
        return_train_score=False,
    )
    return float(np.mean(scores["test_micro_f1"])), float(np.mean(scores["test_macro_f1"]))

In [3]:
data_dir = Path("data") / "cora_centroid"
with open(data_dir / "cora_centroid.pickle", "rb") as f:
    bundle = pickle.load(f)
edge_list = bundle["edge_list"]
y = np.asarray(bundle["labels"], dtype=int)
H = Hypergraph(edge_list)
assert len(y) == len(edge_list), (len(y), len(edge_list))

print(f"Nodes: {len(sorted(H.get_nodes()))}, distinct hyperedges: {len(H.get_edges())}, labels: {y.shape}")

Nodes: 2708, distinct hyperedges: 2590, labels: (2708,)


In [ ]:
# Fixed HMRW / Skip-gram settings (edit here)
fixed_params = dict(
    p=1.0,
    r=1.0,
    num_walks=10,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

q_values = [0.25, 0.5, 1.0, 2.0, 4.0]
n_repeats = 5
base_seed = 42

In [ ]:
rows = []
for q_val in q_values:
    print(q_val)
    micro_runs = []
    macro_runs = []
    for rep in range(n_repeats):
        seed = base_seed + rep
        X = hmrw_embeddings(H, q=q_val, seed=seed, **fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)
    rows.append(
        {
            "q": q_val,
            "micro_f1_mean": float(np.mean(micro_runs)),
            "micro_f1_std": float(np.std(micro_runs)),
            "macro_f1_mean": float(np.mean(macro_runs)),
            "macro_f1_std": float(np.std(macro_runs)),
        }
    )

results = pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)
results_q = results.copy()

# Save results
results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_path = results_dir / "hmrw_q_sweep.csv"
results.to_csv(results_path, index=False)
print(f"Saved results to: {results_path}")

results

In [ ]:
results_q = pd.read_csv(Path("results") / "cora" / "hmrw_q_sweep.csv")
curve = results_q.sort_values("q")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = np.log2(curve["q"].to_numpy())
axes[0].errorbar(x, curve["micro_f1_mean"], yerr=curve["micro_f1_std"], marker="o", capsize=3)
axes[0].set_xlabel(r"$\log_2(q)$")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs q")
axes[1].errorbar(x, curve["macro_f1_mean"], yerr=curve["macro_f1_std"], marker="o", capsize=3, color="tab:orange")
axes[1].set_xlabel(r"$\log_2(q)$")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs q")
fig.tight_layout()
plt.show()

## Additional sweep: vary `p`

This block runs a separate `p` sweep using the currently loaded `H` and `y`, saves a CSV to `results/cora`, and plots Micro/Macro-F1 vs `p`.

In [ ]:
# Separate p sweep (additive; does not modify previous results)
p_fixed_params = dict(
    q=1.0,
    r=1.0,
    num_walks=10,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

p_values = [0.25, 0.5, 1.0, 2.0, 4.0]
n_repeats = 5
base_seed = 42

rows_p = []
for p_val in p_values:
    print(p_val)
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats):
        seed = base_seed + rep
        X = hmrw_embeddings(H, p=p_val, seed=seed, **p_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)
    rows_p.append(
        {
            "p": p_val,
            "micro_f1_mean": float(np.mean(micro_runs)),
            "micro_f1_std": float(np.std(micro_runs)),
            "macro_f1_mean": float(np.mean(macro_runs)),
            "macro_f1_std": float(np.std(macro_runs)),
        }
    )

results_p = pd.DataFrame(rows_p).sort_values("macro_f1_mean", ascending=False)

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_p_path = results_dir / "hmrw_p_sweep.csv"
results_p.to_csv(results_p_path, index=False)
print(f"Saved p results to: {results_p_path}")

results_p

In [ ]:
# Plot p sweep
results_p = pd.read_csv(Path("results") / "cora" / "hmrw_p_sweep.csv")
curve_p = results_p.sort_values("p")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = np.log2(curve_p["p"].to_numpy())

axes[0].errorbar(x, curve_p["micro_f1_mean"], yerr=curve_p["micro_f1_std"], marker="o", capsize=3)
axes[0].set_xlabel(r"$\log_2(p)$")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs p")

axes[1].errorbar(x, curve_p["macro_f1_mean"], yerr=curve_p["macro_f1_std"], marker="o", capsize=3, color="tab:orange")
axes[1].set_xlabel(r"$\log_2(p)$")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs p")

fig.tight_layout()
plt.show()

## Joint plot: `q` and `p` sweeps (2x2)

This compares both sweeps in one figure with four subplots:
- top row: `q` (Micro, Macro)
- bottom row: `p` (Micro, Macro)

It expects `results_q` and `results_p` to already be available.

In [ ]:
# 2x2 joint plot with improved colors
results_q = pd.read_csv(Path("results") / "cora" / "hmrw_q_sweep.csv")
results_p = pd.read_csv(Path("results") / "cora" / "hmrw_p_sweep.csv")
q_curve = results_q.sort_values("q")
p_curve = results_p.sort_values("p")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False)

# Colors chosen for contrast and readability
c_q_micro = "#1f77b4"   # blue
c_q_macro = "#17becf"   # cyan
c_p_micro = "#d62728"   # red
c_p_macro = "#ff7f0e"   # orange

def plot_with_band(ax, x, y, yerr, color, title, xlabel):
    ax.plot(x, y, marker="o", linewidth=2.2, markersize=6, color=color)
    ax.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.18)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("F1 (CV mean)")
    ax.grid(alpha=0.25)

xq = np.log2(q_curve["q"].to_numpy())
plot_with_band(
    axes[0, 0],
    xq,
    q_curve["micro_f1_mean"].to_numpy(),
    q_curve["micro_f1_std"].to_numpy(),
    c_q_micro,
    "Q sweep: Micro-F1",
    r"$\log_2(q)$",
)
plot_with_band(
    axes[0, 1],
    xq,
    q_curve["macro_f1_mean"].to_numpy(),
    q_curve["macro_f1_std"].to_numpy(),
    c_q_macro,
    "Q sweep: Macro-F1",
    r"$\log_2(q)$",
)

xp = np.log2(p_curve["p"].to_numpy())
plot_with_band(
    axes[1, 0],
    xp,
    p_curve["micro_f1_mean"].to_numpy(),
    p_curve["micro_f1_std"].to_numpy(),
    c_p_micro,
    "P sweep: Micro-F1",
    r"$\log_2(p)$",
)
plot_with_band(
    axes[1, 1],
    xp,
    p_curve["macro_f1_mean"].to_numpy(),
    p_curve["macro_f1_std"].to_numpy(),
    c_p_macro,
    "P sweep: Macro-F1",
    r"$\log_2(p)$",
)

# Force the same y-axis range on all panels
all_lower = np.concatenate([
    q_curve["micro_f1_mean"].to_numpy() - q_curve["micro_f1_std"].to_numpy(),
    q_curve["macro_f1_mean"].to_numpy() - q_curve["macro_f1_std"].to_numpy(),
    p_curve["micro_f1_mean"].to_numpy() - p_curve["micro_f1_std"].to_numpy(),
    p_curve["macro_f1_mean"].to_numpy() - p_curve["macro_f1_std"].to_numpy(),
])
all_upper = np.concatenate([
    q_curve["micro_f1_mean"].to_numpy() + q_curve["micro_f1_std"].to_numpy(),
    q_curve["macro_f1_mean"].to_numpy() + q_curve["macro_f1_std"].to_numpy(),
    p_curve["micro_f1_mean"].to_numpy() + p_curve["micro_f1_std"].to_numpy(),
    p_curve["macro_f1_mean"].to_numpy() + p_curve["macro_f1_std"].to_numpy(),
])
ymin = float(all_lower.min())
ymax = float(all_upper.max())
pad = 0.01 * (ymax - ymin if ymax > ymin else 1.0)
from matplotlib.ticker import MultipleLocator, FuncFormatter

for ax in axes.ravel():
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.suptitle("Cora HMRW: Joint view of q and p sweeps", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Grid sweep: `p` × `q`

This section evaluates combinations of `p` and `q` on Cora.

- x-axis: `p`
- y-axis: `q`
- metrics: Micro-F1 and Macro-F1
- repeated runs per setting to estimate mean and std

In [ ]:
pq_fixed_params = dict(
    r=1.0,
    num_walks=10,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

p_grid = [0.25, 0.5, 1.0, 2.0, 4.0]
q_grid = [0.25, 0.5, 1.0, 2.0, 4.0]
n_repeats_pq = 5
base_seed_pq = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_pq_path = results_dir / "hmrw_p_q_grid.csv"

required_cols = [
    "p", "q",
    "micro_f1_mean", "micro_f1_std", "micro_f1_var",
    "macro_f1_mean", "macro_f1_std", "macro_f1_var",
]

if results_pq_path.exists():
    results_pq = pd.read_csv(results_pq_path)
    missing = [c for c in required_cols if c not in results_pq.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_pq["p"] = results_pq["p"].astype(float)
    results_pq["q"] = results_pq["q"].astype(float)
    print(f"Loaded existing file: {results_pq_path} ({len(results_pq)} rows)")
else:
    results_pq = pd.DataFrame(columns=required_cols)
    print(f"Starting new file: {results_pq_path}")

completed = {(float(r.p), float(r.q)) for r in results_pq.itertuples(index=False)}

for p_val in p_grid:
    for q_val in q_grid:
        key = (float(p_val), float(q_val))
        if key in completed:
            print(f"Skip existing pair p={p_val}, q={q_val}")
            continue

        print(f"Run pair p={p_val}, q={q_val}")
        micro_runs, macro_runs = [], []
        for rep in range(n_repeats_pq):
            seed = base_seed_pq + rep
            X = hmrw_embeddings(H, p=p_val, q=q_val, seed=seed, **pq_fixed_params)
            micro, macro = evaluate_embeddings(X, y, seed=seed)
            micro_runs.append(micro)
            macro_runs.append(macro)

        row = {
            "p": float(p_val),
            "q": float(q_val),
            "micro_f1_mean": float(np.mean(micro_runs)),
            "micro_f1_std": float(np.std(micro_runs)),
            "micro_f1_var": float(np.var(micro_runs)),
            "macro_f1_mean": float(np.mean(macro_runs)),
            "macro_f1_std": float(np.std(macro_runs)),
            "macro_f1_var": float(np.var(macro_runs)),
        }

        results_pq = pd.concat([results_pq, pd.DataFrame([row])], ignore_index=True)
        results_pq = results_pq.sort_values(["p", "q"]).drop_duplicates(
            subset=["p", "q"], keep="last"
        )
        results_pq.to_csv(results_pq_path, index=False)
        completed.add(key)
        print(f"Saved progress: {len(results_pq)} rows")

results_pq = results_pq.sort_values(["macro_f1_mean", "micro_f1_mean"], ascending=False)
print(f"Done. Final rows: {len(results_pq)}")
results_pq.head(10)

In [ ]:
p_grid = [0.25, 0.5, 1.0, 2.0, 4.0]
q_grid = [0.25, 0.5, 1.0, 2.0, 4.0]

# Bubble plots: color = mean score, marker size = variance
results_pq = pd.read_csv(Path("results") / "cora" / "hmrw_p_q_grid.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

# Normalize variance to a visible marker-size range
v_micro = results_pq["micro_f1_var"].to_numpy()
v_macro = results_pq["macro_f1_var"].to_numpy()

def to_marker_size(v, smin=120, smax=900):
    vmin, vmax = float(np.min(v)), float(np.max(v))
    if vmax <= vmin:
        return np.full_like(v, (smin + smax) / 2.0, dtype=float)
    return smin + (v - vmin) * (smax - smin) / (vmax - vmin)

size_micro = to_marker_size(v_micro)
size_macro = to_marker_size(v_macro)

sc0 = axes[0].scatter(
    results_pq["p"],
    results_pq["q"],
    c=results_pq["micro_f1_mean"],
    s=size_micro,
    cmap="viridis",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.6,
)
axes[0].set_title("Micro-F1: color=mean, size=variance")
axes[0].set_xlabel("p")
axes[0].set_ylabel("q")
axes[0].set_xscale("log", base=2)
axes[0].set_yscale("log", base=2)
axes[0].set_xticks(p_grid, [str(v) for v in p_grid])
axes[0].set_yticks(q_grid, [str(v) for v in q_grid])
axes[0].grid(alpha=0.2)
fig.colorbar(sc0, ax=axes[0], fraction=0.046, label="Micro-F1 mean")

sc1 = axes[1].scatter(
    results_pq["p"],
    results_pq["q"],
    c=results_pq["macro_f1_mean"],
    s=size_macro,
    cmap="magma",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.6,
)
axes[1].set_title("Macro-F1: color=mean, size=variance")
axes[1].set_xlabel("p")
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log", base=2)
axes[1].set_xticks(p_grid, [str(v) for v in p_grid])
axes[1].set_yticks(q_grid, [str(v) for v in q_grid])
axes[1].grid(alpha=0.2)
fig.colorbar(sc1, ax=axes[1], fraction=0.046, label="Macro-F1 mean")

fig.suptitle("Cora HMRW grid sweep: bubble plots", y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
# Heatmaps: mean and std for Micro/Macro-F1
results_pq = pd.read_csv(Path("results") / "cora" / "hmrw_p_q_grid.csv")
micro_mean = results_pq.pivot(index="q", columns="p", values="micro_f1_mean").sort_index().sort_index(axis=1)
micro_std = results_pq.pivot(index="q", columns="p", values="micro_f1_std").sort_index().sort_index(axis=1)
macro_mean = results_pq.pivot(index="q", columns="p", values="macro_f1_mean").sort_index().sort_index(axis=1)
macro_std = results_pq.pivot(index="q", columns="p", values="macro_f1_std").sort_index().sort_index(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)

im00 = axes[0, 0].imshow(micro_mean.values, aspect="auto", cmap="viridis")
axes[0, 0].set_title("Micro-F1 mean")
fig.colorbar(im00, ax=axes[0, 0], fraction=0.046)

im01 = axes[0, 1].imshow(micro_std.values, aspect="auto", cmap="Blues")
axes[0, 1].set_title("Micro-F1 std")
fig.colorbar(im01, ax=axes[0, 1], fraction=0.046)

im10 = axes[1, 0].imshow(macro_mean.values, aspect="auto", cmap="magma")
axes[1, 0].set_title("Macro-F1 mean")
fig.colorbar(im10, ax=axes[1, 0], fraction=0.046)

im11 = axes[1, 1].imshow(macro_std.values, aspect="auto", cmap="Oranges")
axes[1, 1].set_title("Macro-F1 std")
fig.colorbar(im11, ax=axes[1, 1], fraction=0.046)

for ax in axes.ravel():
    ax.set_xlabel("p")
    ax.set_ylabel("q")
    ax.set_xticks(range(len(micro_mean.columns)), [str(v) for v in micro_mean.columns])
    ax.set_yticks(range(len(micro_mean.index)), [str(v) for v in micro_mean.index])

fig.suptitle("Cora HMRW grid sweep: heatmaps", y=1.01)
fig.tight_layout()
plt.show()

## Sweep over `r` with `p = 2.0`, `q = 2.0`

This matches the updated `hmrw_embeddings(p, q, r, ...)` API.

- fixed: `p=2.0`, `q=2.0`
- sweep: `r`
- resumable run: load existing CSV and skip completed `r` values
- save progress after each `r` iteration

In [ ]:
# Resumable r sweep: p=q=2.0
r_fixed_params = dict(
    p=2.0,
    q=2.0,
    num_walks=10,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

r_values = [0.25, 0.5, 1.0, 2.0, 4.0]
n_repeats_r = 5
base_seed_r = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_r_path = results_dir / "hmrw_r_sweep_p2_q2.csv"

required_cols_r = ["r", "micro_f1_mean", "micro_f1_std", "micro_f1_var", "macro_f1_mean", "macro_f1_std", "macro_f1_var"]
if results_r_path.exists():
    results_r = pd.read_csv(results_r_path)
    missing = [c for c in required_cols_r if c not in results_r.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_r["r"] = results_r["r"].astype(float)
    print(f"Loaded existing r-sweep file: {results_r_path} ({len(results_r)} rows)")
else:
    results_r = pd.DataFrame(columns=required_cols_r)
    print(f"Starting new r-sweep file: {results_r_path}")

done_r = {float(v) for v in results_r["r"].tolist()}

for r_val in r_values:
    if float(r_val) in done_r:
        print(f"Skip existing r={r_val}")
        continue

    print(f"Run r={r_val}")
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats_r):
        seed = base_seed_r + rep
        X = hmrw_embeddings(H, r=r_val, seed=seed, **r_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    new_row = {
        "r": float(r_val),
        "micro_f1_mean": float(np.mean(micro_runs)),
        "micro_f1_std": float(np.std(micro_runs)),
        "micro_f1_var": float(np.var(micro_runs)),
        "macro_f1_mean": float(np.mean(macro_runs)),
        "macro_f1_std": float(np.std(macro_runs)),
        "macro_f1_var": float(np.var(macro_runs)),
    }

    results_r = pd.concat([results_r, pd.DataFrame([new_row])], ignore_index=True)
    results_r = results_r.sort_values("r").drop_duplicates(subset=["r"], keep="last")
    results_r.to_csv(results_r_path, index=False)
    done_r.add(float(r_val))
    print(f"Saved progress: {len(results_r)} rows")

results_r = results_r.sort_values("macro_f1_mean", ascending=False)
print(f"Done r sweep. Total rows: {len(results_r)}")
results_r

In [ ]:
# R sweep plots (p=q fixed): variance clouds + formatted y-axis
from matplotlib.ticker import MultipleLocator, FuncFormatter

results_r = pd.read_csv(Path("results") / "cora" / "hmrw_r_sweep_p2_q2.csv")
curve_r = results_r.sort_values("r")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = np.log2(curve_r["r"].to_numpy())

# Micro-F1
m_mean = curve_r["micro_f1_mean"].to_numpy()
m_std = curve_r["micro_f1_std"].to_numpy()
axes[0].plot(x, m_mean, marker="o", linewidth=2.2, color="tab:blue")
axes[0].fill_between(x, m_mean - m_std, m_mean + m_std, color="tab:blue", alpha=0.18)
axes[0].set_xlabel(r"$\log_2(r)$")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs r")
axes[0].grid(alpha=0.25)

# Macro-F1
M_mean = curve_r["macro_f1_mean"].to_numpy()
M_std = curve_r["macro_f1_std"].to_numpy()
axes[1].plot(x, M_mean, marker="o", linewidth=2.2, color="tab:orange")
axes[1].fill_between(x, M_mean - M_std, M_mean + M_std, color="tab:orange", alpha=0.18)
axes[1].set_xlabel(r"$\log_2(r)$")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs r")
axes[1].grid(alpha=0.25)

# Two-decimal y-ticks without trailing zero
for ax in axes:
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.tight_layout()
plt.show()

In [ ]:
# Resumable r sweep: p=q=2.0
r_fixed_params = dict(
    p=1.0,
    q=1.0,
    num_walks=10,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

r_values = [0.25, 0.5, 1.0, 2.0, 4.0]
n_repeats_r = 5
base_seed_r = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_r_path = results_dir / "hmrw_r_sweep.csv"

required_cols_r = ["r", "micro_f1_mean", "micro_f1_std", "micro_f1_var", "macro_f1_mean", "macro_f1_std", "macro_f1_var"]
if results_r_path.exists():
    results_r = pd.read_csv(results_r_path)
    missing = [c for c in required_cols_r if c not in results_r.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_r["r"] = results_r["r"].astype(float)
    print(f"Loaded existing r-sweep file: {results_r_path} ({len(results_r)} rows)")
else:
    results_r = pd.DataFrame(columns=required_cols_r)
    print(f"Starting new r-sweep file: {results_r_path}")

done_r = {float(v) for v in results_r["r"].tolist()}

for r_val in r_values:
    if float(r_val) in done_r:
        print(f"Skip existing r={r_val}")
        continue

    print(f"Run r={r_val}")
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats_r):
        seed = base_seed_r + rep
        X = hmrw_embeddings(H, r=r_val, seed=seed, **r_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    new_row = {
        "r": float(r_val),
        "micro_f1_mean": float(np.mean(micro_runs)),
        "micro_f1_std": float(np.std(micro_runs)),
        "micro_f1_var": float(np.var(micro_runs)),
        "macro_f1_mean": float(np.mean(macro_runs)),
        "macro_f1_std": float(np.std(macro_runs)),
        "macro_f1_var": float(np.var(macro_runs)),
    }

    results_r = pd.concat([results_r, pd.DataFrame([new_row])], ignore_index=True)
    results_r = results_r.sort_values("r").drop_duplicates(subset=["r"], keep="last")
    results_r.to_csv(results_r_path, index=False)
    done_r.add(float(r_val))
    print(f"Saved progress: {len(results_r)} rows")

results_r = results_r.sort_values("macro_f1_mean", ascending=False)
print(f"Done r sweep. Total rows: {len(results_r)}")
results_r

In [ ]:
# R sweep plots (p=q fixed): variance clouds + formatted y-axis
from matplotlib.ticker import MultipleLocator, FuncFormatter

results_r = pd.read_csv(Path("results") / "cora" / "hmrw_r_sweep.csv")
curve_r = results_r.sort_values("r")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = np.log2(curve_r["r"].to_numpy())

# Micro-F1
m_mean = curve_r["micro_f1_mean"].to_numpy()
m_std = curve_r["micro_f1_std"].to_numpy()
axes[0].plot(x, m_mean, marker="o", linewidth=2.2, color="tab:blue")
axes[0].fill_between(x, m_mean - m_std, m_mean + m_std, color="tab:blue", alpha=0.18)
axes[0].set_xlabel(r"$\log_2(r)$")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs r")
axes[0].grid(alpha=0.25)

# Macro-F1
M_mean = curve_r["macro_f1_mean"].to_numpy()
M_std = curve_r["macro_f1_std"].to_numpy()
axes[1].plot(x, M_mean, marker="o", linewidth=2.2, color="tab:orange")
axes[1].fill_between(x, M_mean - M_std, M_mean + M_std, color="tab:orange", alpha=0.18)
axes[1].set_xlabel(r"$\log_2(r)$")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs r")
axes[1].grid(alpha=0.25)

# Two-decimal y-ticks without trailing zero
for ax in axes:
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.tight_layout()
plt.show()

## Sweep `num_walks` with `p=2`, `q=2`

Resumable experiment:
- fixed: `p=2.0`, `q=2.0`, `r=1.0`
- sweep: `num_walks`
- saves after each `num_walks` value
- skips already completed values if CSV exists

In [ ]:
# Resumable num_walks sweep: p=2, q=2, r=1
nw_fixed_params = dict(
    p=2.0,
    q=2.0,
    r=1.0,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

num_walks_values = [2, 5, 10, 20, 40]
n_repeats_nw = 5
base_seed_nw = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_nw_path = results_dir / "hmrw_num_walks_sweep_p2_q2_r1.csv"

required_cols_nw = [
    "num_walks",
    "micro_f1_mean", "micro_f1_std", "micro_f1_var",
    "macro_f1_mean", "macro_f1_std", "macro_f1_var",
]

if results_nw_path.exists():
    results_nw = pd.read_csv(results_nw_path)
    missing = [c for c in required_cols_nw if c not in results_nw.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_nw["num_walks"] = results_nw["num_walks"].astype(int)
    print(f"Loaded existing file: {results_nw_path} ({len(results_nw)} rows)")
else:
    results_nw = pd.DataFrame(columns=required_cols_nw)
    print(f"Starting new file: {results_nw_path}")

done_nw = {int(v) for v in results_nw["num_walks"].tolist()}

for nw in num_walks_values:
    if int(nw) in done_nw:
        print(f"Skip existing num_walks={nw}")
        continue

    print(f"Run num_walks={nw}")
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats_nw):
        seed = base_seed_nw + rep
        X = hmrw_embeddings(H, num_walks=nw, seed=seed, **nw_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    row = {
        "num_walks": int(nw),
        "micro_f1_mean": float(np.mean(micro_runs)),
        "micro_f1_std": float(np.std(micro_runs)),
        "micro_f1_var": float(np.var(micro_runs)),
        "macro_f1_mean": float(np.mean(macro_runs)),
        "macro_f1_std": float(np.std(macro_runs)),
        "macro_f1_var": float(np.var(macro_runs)),
    }

    results_nw = pd.concat([results_nw, pd.DataFrame([row])], ignore_index=True)
    results_nw = results_nw.sort_values("num_walks").drop_duplicates(subset=["num_walks"], keep="last")
    results_nw.to_csv(results_nw_path, index=False)
    done_nw.add(int(nw))
    print(f"Saved progress: {len(results_nw)} rows")

results_nw = results_nw.sort_values("macro_f1_mean", ascending=False)
print(f"Done num_walks sweep. Total rows: {len(results_nw)}")
results_nw

In [ ]:
# Plot num_walks sweep (read from saved CSV)
from matplotlib.ticker import MultipleLocator, FuncFormatter

results_nw = pd.read_csv(Path("results") / "cora" / "hmrw_num_walks_sweep_p2_q2_r1.csv")
curve_nw = results_nw.sort_values("num_walks")
x = curve_nw["num_walks"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)

m_mean = curve_nw["micro_f1_mean"].to_numpy()
m_std = curve_nw["micro_f1_std"].to_numpy()
axes[0].plot(x, m_mean, marker="o", linewidth=2.2, color="tab:blue")
axes[0].fill_between(x, m_mean - m_std, m_mean + m_std, color="tab:blue", alpha=0.18)
axes[0].set_xlabel("num_walks")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs num_walks")
axes[0].grid(alpha=0.25)

M_mean = curve_nw["macro_f1_mean"].to_numpy()
M_std = curve_nw["macro_f1_std"].to_numpy()
axes[1].plot(x, M_mean, marker="o", linewidth=2.2, color="tab:orange")
axes[1].fill_between(x, M_mean - M_std, M_mean + M_std, color="tab:orange", alpha=0.18)
axes[1].set_xlabel("num_walks")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs num_walks")
axes[1].grid(alpha=0.25)

for ax in axes:
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.tight_layout()
plt.show()

## Sweep `walk_length` with `p=2`, `q=2`

Resumable experiment:
- fixed: `p=2.0`, `q=2.0`, `r=1.0`, `num_walks=10`
- sweep: `walk_length`
- saves after each `walk_length`
- skips already completed values if CSV exists

In [ ]:
# Resumable walk_length sweep: p=2, q=2, r=1, num_walks=10
wl_fixed_params = dict(
    p=2.0,
    q=2.0,
    r=1.0,
    num_walks=10,
    window=10,
    dim=128,
    workers=1,
)

walk_length_values = [10, 20, 40, 80, 120]
n_repeats_wl = 5
base_seed_wl = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_wl_path = results_dir / "hmrw_walk_length_sweep_p2_q2_r1_nw10.csv"

required_cols_wl = [
    "walk_length",
    "micro_f1_mean", "micro_f1_std", "micro_f1_var",
    "macro_f1_mean", "macro_f1_std", "macro_f1_var",
]

if results_wl_path.exists():
    results_wl = pd.read_csv(results_wl_path)
    missing = [c for c in required_cols_wl if c not in results_wl.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_wl["walk_length"] = results_wl["walk_length"].astype(int)
    print(f"Loaded existing file: {results_wl_path} ({len(results_wl)} rows)")
else:
    results_wl = pd.DataFrame(columns=required_cols_wl)
    print(f"Starting new file: {results_wl_path}")

done_wl = {int(v) for v in results_wl["walk_length"].tolist()}

for wl in walk_length_values:
    if int(wl) in done_wl:
        print(f"Skip existing walk_length={wl}")
        continue

    print(f"Run walk_length={wl}")
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats_wl):
        seed = base_seed_wl + rep
        X = hmrw_embeddings(H, walk_length=wl, seed=seed, **wl_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    row = {
        "walk_length": int(wl),
        "micro_f1_mean": float(np.mean(micro_runs)),
        "micro_f1_std": float(np.std(micro_runs)),
        "micro_f1_var": float(np.var(micro_runs)),
        "macro_f1_mean": float(np.mean(macro_runs)),
        "macro_f1_std": float(np.std(macro_runs)),
        "macro_f1_var": float(np.var(macro_runs)),
    }

    results_wl = pd.concat([results_wl, pd.DataFrame([row])], ignore_index=True)
    results_wl = results_wl.sort_values("walk_length").drop_duplicates(subset=["walk_length"], keep="last")
    results_wl.to_csv(results_wl_path, index=False)
    done_wl.add(int(wl))
    print(f"Saved progress: {len(results_wl)} rows")

results_wl = results_wl.sort_values("macro_f1_mean", ascending=False)
print(f"Done walk_length sweep. Total rows: {len(results_wl)}")
results_wl

In [ ]:
# Plot walk_length sweep (read from saved CSV)
from matplotlib.ticker import MultipleLocator, FuncFormatter

results_wl = pd.read_csv(Path("results") / "cora" / "hmrw_walk_length_sweep_p2_q2_r1_nw10.csv")
curve_wl = results_wl.sort_values("walk_length")
x = curve_wl["walk_length"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)

m_mean = curve_wl["micro_f1_mean"].to_numpy()
m_std = curve_wl["micro_f1_std"].to_numpy()
axes[0].plot(x, m_mean, marker="o", linewidth=2.2, color="tab:blue")
axes[0].fill_between(x, m_mean - m_std, m_mean + m_std, color="tab:blue", alpha=0.18)
axes[0].set_xlabel("walk_length")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs walk_length")
axes[0].grid(alpha=0.25)

M_mean = curve_wl["macro_f1_mean"].to_numpy()
M_std = curve_wl["macro_f1_std"].to_numpy()
axes[1].plot(x, M_mean, marker="o", linewidth=2.2, color="tab:orange")
axes[1].fill_between(x, M_mean - M_std, M_mean + M_std, color="tab:orange", alpha=0.18)
axes[1].set_xlabel("walk_length")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs walk_length")
axes[1].grid(alpha=0.25)

for ax in axes:
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.tight_layout()
plt.show()

In [ ]:
# Resumable walk_length sweep: p=2, q=2, r=1, num_walks=10
wl_fixed_params = dict(
    p=2.0,
    q=2.0,
    r=1.0,
    num_walks=5,
    window=10,
    dim=128,
    workers=1,
)

walk_length_values = [10, 20, 40, 80, 120]
n_repeats_wl = 5
base_seed_wl = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_wl_path = results_dir / "hmrw_walk_length_sweep_p2_q2_r1_nw5.csv"

required_cols_wl = [
    "walk_length",
    "micro_f1_mean", "micro_f1_std", "micro_f1_var",
    "macro_f1_mean", "macro_f1_std", "macro_f1_var",
]

if results_wl_path.exists():
    results_wl = pd.read_csv(results_wl_path)
    missing = [c for c in required_cols_wl if c not in results_wl.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_wl["walk_length"] = results_wl["walk_length"].astype(int)
    print(f"Loaded existing file: {results_wl_path} ({len(results_wl)} rows)")
else:
    results_wl = pd.DataFrame(columns=required_cols_wl)
    print(f"Starting new file: {results_wl_path}")

done_wl = {int(v) for v in results_wl["walk_length"].tolist()}

for wl in walk_length_values:
    if int(wl) in done_wl:
        print(f"Skip existing walk_length={wl}")
        continue

    print(f"Run walk_length={wl}")
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats_wl):
        seed = base_seed_wl + rep
        X = hmrw_embeddings(H, walk_length=wl, seed=seed, **wl_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    row = {
        "walk_length": int(wl),
        "micro_f1_mean": float(np.mean(micro_runs)),
        "micro_f1_std": float(np.std(micro_runs)),
        "micro_f1_var": float(np.var(micro_runs)),
        "macro_f1_mean": float(np.mean(macro_runs)),
        "macro_f1_std": float(np.std(macro_runs)),
        "macro_f1_var": float(np.var(macro_runs)),
    }

    results_wl = pd.concat([results_wl, pd.DataFrame([row])], ignore_index=True)
    results_wl = results_wl.sort_values("walk_length").drop_duplicates(subset=["walk_length"], keep="last")
    results_wl.to_csv(results_wl_path, index=False)
    done_wl.add(int(wl))
    print(f"Saved progress: {len(results_wl)} rows")

results_wl = results_wl.sort_values("macro_f1_mean", ascending=False)
print(f"Done walk_length sweep. Total rows: {len(results_wl)}")
results_wl

In [ ]:
# Plot walk_length sweep (read from saved CSV)
from matplotlib.ticker import MultipleLocator, FuncFormatter

results_wl = pd.read_csv(Path("results") / "cora" / "hmrw_walk_length_sweep_p2_q2_r1_nw5.csv")
curve_wl = results_wl.sort_values("walk_length")
x = curve_wl["walk_length"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)

m_mean = curve_wl["micro_f1_mean"].to_numpy()
m_std = curve_wl["micro_f1_std"].to_numpy()
axes[0].plot(x, m_mean, marker="o", linewidth=2.2, color="tab:blue")
axes[0].fill_between(x, m_mean - m_std, m_mean + m_std, color="tab:blue", alpha=0.18)
axes[0].set_xlabel("walk_length")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs walk_length")
axes[0].grid(alpha=0.25)

M_mean = curve_wl["macro_f1_mean"].to_numpy()
M_std = curve_wl["macro_f1_std"].to_numpy()
axes[1].plot(x, M_mean, marker="o", linewidth=2.2, color="tab:orange")
axes[1].fill_between(x, M_mean - M_std, M_mean + M_std, color="tab:orange", alpha=0.18)
axes[1].set_xlabel("walk_length")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs walk_length")
axes[1].grid(alpha=0.25)

for ax in axes:
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.tight_layout()
plt.show()

## Sweep embedding dimension (`dim`) first

Use the same defaults as the joint section and only vary `dim` to decide whether it should be included in the broader hyperparameter sensitivity figure.

Defaults:
- `p = 1.0`, `q = 1.0`, `r = 1.0`
- `num_walks = 5`, `walk_length = 20`
- `window = 10`, `workers = 1`

In [ ]:
# Resumable dim sweep with default settings

dim_fixed_params = dict(
    p=1.0,
    q=1.0,
    r=1.0,
    num_walks=5,
    walk_length=20,
    window=10,
    workers=1,
)

dim_values = [16, 32, 64, 128, 256, 512, 1024]
n_repeats_dim = 5
base_seed_dim = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_dim_path = results_dir / "hmrw_dim_sweep.csv"

required_cols_dim = [
    "dim",
    "micro_f1_mean", "micro_f1_std", "micro_f1_var",
    "macro_f1_mean", "macro_f1_std", "macro_f1_var",
]

if results_dim_path.exists():
    results_dim = pd.read_csv(results_dim_path)
    missing = [c for c in required_cols_dim if c not in results_dim.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    results_dim["dim"] = results_dim["dim"].astype(int)
    print(f"Loaded existing file: {results_dim_path} ({len(results_dim)} rows)")
else:
    results_dim = pd.DataFrame(columns=required_cols_dim)
    print(f"Starting new file: {results_dim_path}")

done_dim = {int(v) for v in results_dim["dim"].tolist()}

for dim_val in dim_values:
    if int(dim_val) in done_dim:
        print(f"Skip existing dim={dim_val}")
        continue

    print(f"Run dim={dim_val}")
    micro_runs, macro_runs = [], []
    for rep in range(n_repeats_dim):
        seed = base_seed_dim + rep
        X = hmrw_embeddings(H, dim=int(dim_val), seed=seed, **dim_fixed_params)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    row = {
        "dim": int(dim_val),
        "micro_f1_mean": float(np.mean(micro_runs)),
        "micro_f1_std": float(np.std(micro_runs)),
        "micro_f1_var": float(np.var(micro_runs)),
        "macro_f1_mean": float(np.mean(macro_runs)),
        "macro_f1_std": float(np.std(macro_runs)),
        "macro_f1_var": float(np.var(macro_runs)),
    }

    results_dim = pd.concat([results_dim, pd.DataFrame([row])], ignore_index=True)
    results_dim = results_dim.sort_values("dim").drop_duplicates(subset=["dim"], keep="last")
    results_dim.to_csv(results_dim_path, index=False)
    done_dim.add(int(dim_val))
    print(f"Saved progress: {len(results_dim)} rows")

results_dim = results_dim.sort_values("macro_f1_mean", ascending=False)
print(f"Done dim sweep. Total rows: {len(results_dim)}")
results_dim

In [ ]:
# Plot dim sweep from saved CSV
from matplotlib.ticker import MultipleLocator, FuncFormatter

results_dim = pd.read_csv(Path("results") / "cora" / "hmrw_dim_sweep.csv")
curve_dim = results_dim.sort_values("dim")
x = curve_dim["dim"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)

m_mean = curve_dim["micro_f1_mean"].to_numpy()
m_std = curve_dim["micro_f1_std"].to_numpy()
axes[0].plot(x, m_mean, marker="o", linewidth=2.2, color="tab:blue")
axes[0].fill_between(x, m_mean - m_std, m_mean + m_std, color="tab:blue", alpha=0.18)
axes[0].set_xlabel("Embedding dimension (dim)")
axes[0].set_ylabel("Micro-F1 (CV mean)")
axes[0].set_title("Micro-F1 vs dim")
axes[0].grid(alpha=0.25)

M_mean = curve_dim["macro_f1_mean"].to_numpy()
M_std = curve_dim["macro_f1_std"].to_numpy()
axes[1].plot(x, M_mean, marker="o", linewidth=2.2, color="tab:orange")
axes[1].fill_between(x, M_mean - M_std, M_mean + M_std, color="tab:orange", alpha=0.18)
axes[1].set_xlabel("Embedding dimension (dim)")
axes[1].set_ylabel("Macro-F1 (CV mean)")
axes[1].set_title("Macro-F1 vs dim")
axes[1].grid(alpha=0.25)

for ax in axes:
    ax.yaxis.set_major_locator(MultipleLocator(0.01))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.tight_layout()
plt.show()

## Joint hyperparameter sensitivity (single figure, resumable)

Defaults (used for non-swept parameters):
- `p = 1.0`
- `q = 1.0`
- `r = 1.0`
- `num_walks = 5`
- `walk_length = 20`
- `window = 10`
- `dim = 256`
- `workers = 1`

Each sweep varies one hyperparameter while keeping the others at defaults.
Results are saved incrementally and can resume after interruption.

In [ ]:
# Resumable sweeps for multiple hyperparameters (incremental save)

from pathlib import Path
import pandas as pd
import numpy as np

base_params = dict(
    p=1.0,
    q=1.0,
    r=1.0,
    num_walks=5,
    walk_length=20,
    window=10,
    dim=256,
    workers=1,
)

sweep_config = {
    "p": [0.25, 0.5, 1.0, 2.0, 4.0],
    "q": [0.25, 0.5, 1.0, 2.0, 4.0],
    "r": [0.25, 0.5, 1.0, 2.0, 4.0],
    "num_walks": [2, 5, 10, 20, 40],
    "walk_length": [10, 20, 40, 80],
    "dim": [16, 64, 128, 256, 512],
}

n_repeats_joint = 10
base_seed_joint = 42

results_dir = Path("results") / "cora"
results_dir.mkdir(parents=True, exist_ok=True)
results_joint_path = results_dir / "hmrw_joint_sensitivity.csv"

required_cols = [
    "hyperparameter", "value",
    "micro_f1_mean", "micro_f1_std", "micro_f1_var",
    "macro_f1_mean", "macro_f1_std", "macro_f1_var",
]

if results_joint_path.exists():
    results_joint = pd.read_csv(results_joint_path)
    missing = [c for c in required_cols if c not in results_joint.columns]
    if missing:
        raise ValueError(f"Existing file missing columns: {missing}")
    print(f"Loaded existing file: {results_joint_path} ({len(results_joint)} rows)")
else:
    results_joint = pd.DataFrame(columns=required_cols)
    print(f"Starting new file: {results_joint_path}")

# Build completed key set
completed = {
    (str(r.hyperparameter), float(r.value))
    for r in results_joint.itertuples(index=False)
}

for hp_name, hp_values in sweep_config.items():
    for hp_val in hp_values:
        key = (hp_name, float(hp_val))
        if key in completed:
            print(f"Skip existing {hp_name}={hp_val}")
            continue

        params = dict(base_params)
        if hp_name in {"num_walks", "walk_length", "window", "dim", "workers"}:
            params[hp_name] = int(hp_val)
        else:
            params[hp_name] = float(hp_val)

        print(f"Run {hp_name}={hp_val} with defaults for others")
        micro_runs, macro_runs = [], []
        for rep in range(n_repeats_joint):
            seed = base_seed_joint + rep
            X = hmrw_embeddings(H, seed=seed, **params)
            micro, macro = evaluate_embeddings(X, y, seed=seed)
            micro_runs.append(micro)
            macro_runs.append(macro)

        row = {
            "hyperparameter": hp_name,
            "value": float(hp_val),
            "micro_f1_mean": float(np.mean(micro_runs)),
            "micro_f1_std": float(np.std(micro_runs)),
            "micro_f1_var": float(np.var(micro_runs)),
            "macro_f1_mean": float(np.mean(macro_runs)),
            "macro_f1_std": float(np.std(macro_runs)),
            "macro_f1_var": float(np.var(macro_runs)),
        }

        # Save after each completed sweep point
        results_joint = pd.concat([results_joint, pd.DataFrame([row])], ignore_index=True)
        results_joint = results_joint.sort_values(["hyperparameter", "value"]).drop_duplicates(
            subset=["hyperparameter", "value"], keep="last"
        )
        results_joint.to_csv(results_joint_path, index=False)
        completed.add(key)
        print(f"Saved progress: {len(results_joint)} rows")

print(f"Done. Total rows in joint file: {len(results_joint)}")
results_joint.head(15)

In [ ]:
# Joint plot (single figure) from saved CSV
from matplotlib.ticker import MultipleLocator, FuncFormatter
import matplotlib.pyplot as plt

results_joint = pd.read_csv(Path("results") / "cora" / "hmrw_joint_sensitivity.csv")

# Include dim by merging dim-sweep results into the same plotting frame
dim_path = Path("results") / "cora" / "hmrw_dim_sweep_defaults.csv"
if dim_path.exists():
    dim_df = pd.read_csv(dim_path)
    dim_df = dim_df.rename(columns={"dim": "value"})
    dim_df.insert(0, "hyperparameter", "dim")
    dim_df = dim_df[[
        "hyperparameter", "value",
        "micro_f1_mean", "micro_f1_std", "micro_f1_var",
        "macro_f1_mean", "macro_f1_std", "macro_f1_var",
    ]]
    results_joint = pd.concat([results_joint, dim_df], ignore_index=True)
    results_joint = results_joint.drop_duplicates(subset=["hyperparameter", "value"], keep="last")

# Hide dim=16 from the final joint plot
results_joint = results_joint[~((results_joint["hyperparameter"] == "dim") & (results_joint["value"] < 70))]

plot_order = ["p", "q", "r", "num_walks", "walk_length", "dim"]
plot_label = {
    "p": r"$\log_2 p$",
    "q": r"$\log_2 q$",
    "r": r"$\log_2 r$",
    "num_walks": "Number of walks per node",
    "walk_length": "Length of walk",
    "dim": "Embedding dimension",
}

fig, axes = plt.subplots(2, len(plot_order), figsize=(3.6 * len(plot_order), 7), sharey=True)

for j, hp in enumerate(plot_order):
    df = results_joint[results_joint["hyperparameter"] == hp].copy().sort_values("value")
    if df.empty:
        axes[0, j].set_title(f"{hp} (no data)")
        axes[0, j].axis("off")
        axes[1, j].axis("off")
        continue

    x_raw = df["value"].to_numpy(dtype=float)
    x = np.log2(x_raw) if hp in {"p", "q", "r"} else x_raw

    micro_mean = df["micro_f1_mean"].to_numpy()
    micro_std = df["micro_f1_std"].to_numpy()
    macro_mean = df["macro_f1_mean"].to_numpy()
    macro_std = df["macro_f1_std"].to_numpy()

    # Top row: Macro-F1
    axes[0, j].plot(x, macro_mean, marker="o", linewidth=2.0, color="#d62728")
    axes[0, j].fill_between(x, macro_mean - macro_std, macro_mean + macro_std, color="#d62728", alpha=0.18)
    axes[0, j].set_title(hp)
    axes[0, j].set_xlabel(plot_label[hp])
    axes[0, j].grid(alpha=0.25)

    # Bottom row: Micro-F1
    axes[1, j].plot(x, micro_mean, marker="o", linewidth=2.0, color="#1f77b4")
    axes[1, j].fill_between(x, micro_mean - micro_std, micro_mean + micro_std, color="#1f77b4", alpha=0.18)
    axes[1, j].set_xlabel(plot_label[hp])
    axes[1, j].grid(alpha=0.25)

axes[0, 0].set_ylabel("Macro-F1 score")
axes[1, 0].set_ylabel("Micro-F1 score")

# Shared y formatting (2 decimals max, no trailing zeros)
for row in axes:
    for ax in row:
        ax.yaxis.set_major_locator(MultipleLocator(0.02))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.2f}".rstrip("0").rstrip(".")))

fig.suptitle("Cora HMRW joint hyperparameter sensitivity (defaults: p=1, q=1, r=1, num_walks=5, walk_length=20)", y=1.02)
fig.tight_layout()
plt.show()

## Method comparison: HMRW vs previous methods

Compares your method (`hmrw_embeddings`) with the three earlier random-walk baselines on the same Cora hypergraph and evaluation protocol.

In [4]:
# Compare methods: Hyper2Vec, Clique+Node2Vec, HMRW
from src.utils import hyper2vec_embeddings, clique_node2vec_embeddings, hmrw_embeddings

compare_params = dict(
    num_walks=5,
    walk_length=20,
    window=10,
    dim=128,
    workers=1,
)

methods = {
    "Hyper2Vec": lambda seed: hyper2vec_embeddings(H, p=1.0, q=1.0, r=0.0, seed=seed, **compare_params),
    "Clique+Node2Vec": lambda seed: clique_node2vec_embeddings(H, p=1.0, q=1.0, seed=seed, **compare_params),
    "HMRW": lambda seed: hmrw_embeddings(H, p=2.0, q=2.0, r=1.0, seed=seed, **compare_params),
}

n_repeats_cmp = 5
base_seed_cmp = 42
rows_cmp = []

for method_name, emb_fn in methods.items():
    micro_runs, macro_runs = [], []
    print(method_name)
    for rep in range(n_repeats_cmp):
        print(rep + 1)
        seed = base_seed_cmp + rep
        X = emb_fn(seed)
        micro, macro = evaluate_embeddings(X, y, seed=seed)
        micro_runs.append(micro)
        macro_runs.append(macro)

    rows_cmp.append(
        {
            "method": method_name,
            "micro_f1_mean": float(np.mean(micro_runs)),
            "micro_f1_std": float(np.std(micro_runs)),
            "macro_f1_mean": float(np.mean(macro_runs)),
            "macro_f1_std": float(np.std(macro_runs)),
        }
    )

comparison_df = pd.DataFrame(rows_cmp).sort_values("macro_f1_mean", ascending=False)
comparison_df

Hyper2Vec
1
2
3
4
5
Clique+Node2Vec
1
2
3
4


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


HMRW
1
2
3
4


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


5


,method,micro_f1_mean,micro_f1_std,macro_f1_mean,macro_f1_std
0,Hyper2Vec,0.827545,0.003237,0.818321,0.003426
2,HMRW,0.822304,0.006566,0.813951,0.005897
1,Clique+Node2Vec,0.816908,0.001920,0.804930,0.002249
